# Hospital Silver – Capa Plata
Notebook 1 de 2 de la arquitectura medallón del hospital.

| Capa | Ubicación | Tipo de objeto | Propósito |
|---|---|---|---|
| Bronce (entrada) | `bi.hospital` | Tablas originales (Kaggle) | Datos tal como llegaron |
| **Plata (este notebook)** | `bi.hospital_silver` | Tablas Delta (CTAS) | Datos limpios, tipados, estandarizados y auditables |
| Oro (siguiente) | `bi.hospital_gold` | Vistas | Se construye en el notebook **Hospital Gold** |

**Orden:** ejecutar este notebook completo y revisar las validaciones antes de correr *Hospital Gold*.

## 0. Esquema

In [0]:
CREATE SCHEMA IF NOT EXISTS bi.hospital_silver COMMENT 'Capa plata: datos limpios del hospital';

# CAPA PLATA
Reglas generales aplicadas a todas las tablas:
- Nombres de columnas en `snake_case` y corregidos (`appoIntment_Id` → `appointment_id`, `admisson_ID` → `admission_id`).
- `TRIM` en textos y tipos correctos (`timestamp` a medianoche → `DATE`, montos → `DECIMAL(12,2)`).
- Deduplicación defensiva por llave primaria con `QUALIFY ROW_NUMBER()`.
- Columna de auditoría `dwh_load_date`.

### 1.1 Departamentos, pabellones, habitaciones y camas (estructura física)

In [0]:
CREATE OR REPLACE TABLE bi.hospital_silver.departments
COMMENT 'Departamentos del hospital'
AS
SELECT
    CAST(dept_Id AS BIGINT)  AS dept_id,
    trim(dept_Name)          AS dept_name,
    current_timestamp()      AS dwh_load_date
FROM bi.hospital.department
WHERE dept_Id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY dept_Id ORDER BY dept_Name) = 1;

CREATE OR REPLACE TABLE bi.hospital_silver.wards
COMMENT 'Pabellones; cada uno pertenece a un departamento'
AS
SELECT
    CAST(ward_No AS BIGINT)  AS ward_no,
    trim(ward_Name)          AS ward_name,
    CAST(dept_Id AS BIGINT)  AS dept_id,
    current_timestamp()      AS dwh_load_date
FROM bi.hospital.ward
WHERE ward_No IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY ward_No ORDER BY ward_Name) = 1;

CREATE OR REPLACE TABLE bi.hospital_silver.rooms
COMMENT 'Habitaciones (Standard, Deluxe, Super Deluxe, Operation Theatre)'
AS
SELECT
    CAST(room_No AS BIGINT)  AS room_no,
    CAST(dept_Id AS BIGINT)  AS dept_id,
    trim(room_Type)          AS room_type,
    current_timestamp()      AS dwh_load_date
FROM bi.hospital.room
WHERE room_No IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY room_No ORDER BY room_Type) = 1;

CREATE OR REPLACE TABLE bi.hospital_silver.beds
COMMENT 'Camas; cada una pertenece a un pabellón'
AS
SELECT
    CAST(bed_No  AS BIGINT)  AS bed_no,
    CAST(ward_No AS BIGINT)  AS ward_no,
    current_timestamp()      AS dwh_load_date
FROM bi.hospital.bed
WHERE bed_No IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY bed_No ORDER BY ward_No) = 1;

num_affected_rows,num_inserted_rows


num_affected_rows,num_inserted_rows


num_affected_rows,num_inserted_rows


num_affected_rows,num_inserted_rows


### 1.2 Personas: pacientes, médicos, enfermería y auxiliares
Hallazgos del perfilamiento que se corrigen aquí:
- `doctor.FName` trae el prefijo **"Dr."** (por eso `CONCAT(FName,' ',LName)` daba "Dr. Raza Hasnain"). Se quita y se deja `title`.
- Error de capitalización **"And"** dentro de palabras: `HAnd Surgeon`, `Siri ChAnd`, `ArjamAnd`.
- `surgeon_Type` es NULL en 362 de 400 médicos → `'No aplica'` + bandera `is_surgeon`.
- `contact_No` es `bigint` en pacientes y `string` en médicos → todo pasa a `STRING` (un teléfono no se suma).
- `office_No` viene como `double` → `INT`.
- La dirección del paciente tiene un patrón fijo (`..., Ciudad, Provincia, País`) → se separa en columnas geográficas.

In [0]:
CREATE OR REPLACE TABLE bi.hospital_silver.patients
COMMENT 'Pacientes limpios con geografía derivada de la dirección'
AS
WITH src AS (
    SELECT *, split(pt_Address, ',') AS addr_parts
    FROM bi.hospital.patients
    WHERE patient_Id IS NOT NULL
)
SELECT
    CAST(patient_Id AS BIGINT)                              AS patient_id,
    trim(FName)                                             AS first_name,
    trim(LName)                                             AS last_name,
    concat_ws(' ', trim(FName), trim(LName))                AS full_name,
    CASE upper(trim(Gender)) WHEN 'M' THEN 'M' WHEN 'F' THEN 'F' ELSE 'N/D' END AS gender_code,
    CASE upper(trim(Gender))
         WHEN 'M' THEN 'Masculino'
         WHEN 'F' THEN 'Femenino'
         ELSE 'No definido' END                             AS gender,
    -- Una fecha de nacimiento futura es un error de captura: se anula
    CASE WHEN CAST(Date_Of_Birth AS DATE) > current_date() THEN NULL
         ELSE CAST(Date_Of_Birth AS DATE) END               AS birth_date,
    CAST(contact_No AS STRING)                              AS contact_no,
    trim(pt_Address)                                        AS address,
    CASE WHEN size(addr_parts) >= 3 THEN trim(element_at(addr_parts, -3)) END AS city,
    CASE WHEN size(addr_parts) >= 2 THEN trim(element_at(addr_parts, -2)) END AS province,
    CASE WHEN size(addr_parts) >= 1 THEN trim(element_at(addr_parts, -1)) END AS country,
    current_timestamp()                                     AS dwh_load_date
FROM src
QUALIFY ROW_NUMBER() OVER (PARTITION BY patient_Id ORDER BY patient_Id) = 1;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE bi.hospital_silver.doctors
COMMENT 'Médicos sin prefijo Dr., con capitalización corregida y bandera de cirujano'
AS
WITH src AS (
    SELECT
        *,
        -- Quita "Dr." o "Dr " al inicio (sin tocar nombres como "Drake")
        trim(regexp_replace(FName, '(?i)^\\s*dr(\\.\\s*|\\s+)', '')) AS fname_clean
    FROM bi.hospital.doctor
    WHERE doct_Id IS NOT NULL
)
SELECT
    CAST(doct_Id AS BIGINT)                                         AS doctor_id,
    CAST(dept_Id AS BIGINT)                                         AS dept_id,
    'Dr.'                                                           AS title,
    regexp_replace(fname_clean, '([a-z])And', '$1and')              AS first_name,
    regexp_replace(trim(LName), '([a-z])And', '$1and')              AS last_name,
    concat_ws(' ',
        regexp_replace(fname_clean, '([a-z])And', '$1and'),
        regexp_replace(trim(LName), '([a-z])And', '$1and'))         AS full_name,
    CASE upper(trim(Gender)) WHEN 'M' THEN 'Masculino' WHEN 'F' THEN 'Femenino' ELSE 'No definido' END AS gender,
    trim(CAST(contact_No AS STRING))                                AS contact_no,
    COALESCE(
        regexp_replace(nullif(trim(surgeon_Type), ''), '([a-z])And', '$1and'),
        'No aplica')                                                AS surgeon_type,
    (nullif(trim(surgeon_Type), '') IS NOT NULL)                    AS is_surgeon,
    CAST(office_No AS INT)                                          AS office_no,
    current_timestamp()                                             AS dwh_load_date
FROM src
QUALIFY ROW_NUMBER() OVER (PARTITION BY doct_Id ORDER BY doct_Id) = 1;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE bi.hospital_silver.nurses
COMMENT 'Personal de enfermería'
AS
SELECT
    CAST(nurse_Id AS BIGINT)                        AS nurse_id,
    CAST(dept_Id  AS BIGINT)                        AS dept_id,
    trim(FName)                                     AS first_name,
    trim(LName)                                     AS last_name,
    concat_ws(' ', trim(FName), trim(LName))        AS full_name,
    CASE upper(trim(Gender)) WHEN 'M' THEN 'Masculino' WHEN 'F' THEN 'Femenino' ELSE 'No definido' END AS gender,
    -- En bronce la columna viene mal escrita: conatct_No (typo del dataset original)
    trim(CAST(conatct_No AS STRING))                AS contact_no,
    current_timestamp()                             AS dwh_load_date
FROM bi.hospital.nurse
WHERE nurse_Id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY nurse_Id ORDER BY nurse_Id) = 1;

CREATE OR REPLACE TABLE bi.hospital_silver.helpers
COMMENT 'Personal auxiliar'
AS
SELECT
    CAST(helper_Id AS BIGINT)                       AS helper_id,
    CAST(dept_Id   AS BIGINT)                       AS dept_id,
    trim(FName)                                     AS first_name,
    trim(LName)                                     AS last_name,
    concat_ws(' ', trim(FName), trim(LName))        AS full_name,
    CASE upper(trim(Gender)) WHEN 'M' THEN 'Masculino' WHEN 'F' THEN 'Femenino' ELSE 'No definido' END AS gender,
    trim(CAST(contact_No AS STRING))                AS contact_no,
    current_timestamp()                             AS dwh_load_date
FROM bi.hospital.helpers
WHERE helper_Id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY helper_Id ORDER BY helper_Id) = 1;

num_affected_rows,num_inserted_rows


num_affected_rows,num_inserted_rows


### 1.3 Eventos: citas, estancias, cirugías y registros médicos
- Estado de la cita traducido (`Completed` → `Completada`, `No-Show` → `No asistió`, ...).
- Fechas de egreso anteriores al ingreso se anulan y se calculan `length_of_stay_days` y `is_open` (paciente aún hospitalizado).
- Duración de cirugía en minutos (negativa → NULL).
- Presión arterial `"120/80"` separada en sistólica y diastólica; temperatura convertida a °C.

In [0]:
CREATE OR REPLACE TABLE bi.hospital_silver.appointments
COMMENT 'Citas médicas limpias'
AS
SELECT
    CAST(appoIntment_Id AS BIGINT)                  AS appointment_id,
    CAST(patient_Id     AS BIGINT)                  AS patient_id,
    CAST(doct_Id        AS BIGINT)                  AS doctor_id,
    CAST(appointment_Date AS DATE)                  AS appointment_date,
    trim(reason)                                    AS reason,
    trim(appointment_status)                        AS status_code,
    CASE trim(appointment_status)
         WHEN 'Completed' THEN 'Completada'
         WHEN 'Scheduled' THEN 'Programada'
         WHEN 'Pending'   THEN 'Pendiente'
         WHEN 'Cancelled' THEN 'Cancelada'
         WHEN 'No-Show'   THEN 'No asistió'
         ELSE 'No definido' END                     AS status,
    trim(mode_of_appointment)                       AS appointment_mode,
    trim(mode_of_payment)                           AS payment_method,
    CASE WHEN payment_amount < 0 THEN NULL
         ELSE CAST(payment_amount AS DECIMAL(12,2)) END AS payment_amount,
    current_timestamp()                             AS dwh_load_date
FROM bi.hospital.appointment
WHERE appoIntment_Id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY appoIntment_Id ORDER BY appointment_Date DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE bi.hospital_silver.bed_records
COMMENT 'Estancias en cama'
AS
WITH src AS (
    SELECT *,
           CAST(admission_Date AS DATE) AS adm_d,
           CAST(discharge_Date AS DATE) AS dis_d
    FROM bi.hospital.bedrecords
    WHERE admission_Id IS NOT NULL
)
SELECT
    CAST(admission_Id AS BIGINT)                        AS admission_id,
    CAST(bed_No       AS BIGINT)                        AS bed_no,
    CAST(patient_Id   AS BIGINT)                        AS patient_id,
    CAST(nurse_Id     AS BIGINT)                        AS nurse_id,
    CAST(helper_Id    AS BIGINT)                        AS helper_id,
    adm_d                                               AS admission_date,
    CASE WHEN dis_d < adm_d THEN NULL ELSE dis_d END    AS discharge_date,
    CASE WHEN dis_d >= adm_d THEN datediff(dis_d, adm_d) END AS length_of_stay_days,
    (dis_d IS NULL OR dis_d < adm_d)                    AS is_open,
    CASE WHEN amount < 0 THEN NULL ELSE CAST(amount AS DECIMAL(12,2)) END AS amount,
    trim(mode_of_payment)                               AS payment_method,
    current_timestamp()                                 AS dwh_load_date
FROM src
QUALIFY ROW_NUMBER() OVER (PARTITION BY admission_Id ORDER BY admission_Date DESC) = 1;

CREATE OR REPLACE TABLE bi.hospital_silver.room_records
COMMENT 'Estancias en habitación (corrige admisson_ID → admission_id)'
AS
WITH src AS (
    SELECT *,
           CAST(admission_Date AS DATE) AS adm_d,
           CAST(discharge_Date AS DATE) AS dis_d
    FROM bi.hospital.roomrecords
    WHERE admisson_ID IS NOT NULL
)
SELECT
    CAST(admisson_ID AS BIGINT)                         AS admission_id,
    CAST(room_no     AS BIGINT)                         AS room_no,
    CAST(patient_Id  AS BIGINT)                         AS patient_id,
    CAST(nurse_Id    AS BIGINT)                         AS nurse_id,
    CAST(helper_Id   AS BIGINT)                         AS helper_id,
    adm_d                                               AS admission_date,
    CASE WHEN dis_d < adm_d THEN NULL ELSE dis_d END    AS discharge_date,
    CASE WHEN dis_d >= adm_d THEN datediff(dis_d, adm_d) END AS length_of_stay_days,
    (dis_d IS NULL OR dis_d < adm_d)                    AS is_open,
    CASE WHEN amount < 0 THEN NULL ELSE CAST(amount AS DECIMAL(12,2)) END AS amount,
    trim(mode_of_payment)                               AS payment_method,
    current_timestamp()                                 AS dwh_load_date
FROM src
QUALIFY ROW_NUMBER() OVER (PARTITION BY admisson_ID ORDER BY admission_Date DESC) = 1;

num_affected_rows,num_inserted_rows


num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE bi.hospital_silver.surgery_records
COMMENT 'Cirugías con duración en minutos'
AS
WITH src AS (
    SELECT *, timestampdiff(MINUTE, start_Timestamp, end_Timestamp) AS dur
    FROM bi.hospital.surgeryrecord
    WHERE surgery_Id IS NOT NULL
)
SELECT
    CAST(surgery_Id  AS BIGINT)                 AS surgery_id,
    CAST(patient_Id  AS BIGINT)                 AS patient_id,
    CAST(surgeon_Id  AS BIGINT)                 AS surgeon_id,
    CAST(room_no     AS BIGINT)                 AS room_no,
    CAST(nurse_Id    AS BIGINT)                 AS nurse_id,
    CAST(helper_Id   AS BIGINT)                 AS helper_id,
    trim(surgery_Type)                          AS surgery_type,
    CAST(surgery_Date AS DATE)                  AS surgery_date,
    start_Timestamp                             AS start_ts,
    end_Timestamp                               AS end_ts,
    CASE WHEN dur >= 0 THEN dur END             AS duration_minutes,
    nullif(trim(notes), '')                     AS notes,
    current_timestamp()                         AS dwh_load_date
FROM src
QUALIFY ROW_NUMBER() OVER (PARTITION BY surgery_Id ORDER BY surgery_Date DESC) = 1;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE bi.hospital_silver.medical_records
COMMENT 'Registros clínicos con signos vitales estandarizados'
AS
SELECT
    CAST(record_Id  AS BIGINT)                                  AS record_id,
    CAST(doct_Id    AS BIGINT)                                  AS doctor_id,
    CAST(patient_Id AS BIGINT)                                  AS patient_id,
    CAST(visit_Date AS DATE)                                    AS visit_date,
    CAST(curr_Weight AS DECIMAL(6,2))                           AS weight,
    CAST(curr_height AS DECIMAL(6,2))                           AS height,
    trim(curr_Blood_Pressure)                                   AS blood_pressure,
    -- regexp_extract devuelve '' si no hay coincidencia; try_cast('' AS INT) → NULL
    try_cast(regexp_extract(curr_Blood_Pressure, '^\\s*(\\d+)\\s*/\\s*(\\d+)', 1) AS INT) AS systolic_bp,
    try_cast(regexp_extract(curr_Blood_Pressure, '^\\s*(\\d+)\\s*/\\s*(\\d+)', 2) AS INT) AS diastolic_bp,
    CAST(curr_Temp_F AS DECIMAL(5,1))                           AS temp_f,
    CAST(round((curr_Temp_F - 32) * 5 / 9, 1) AS DECIMAL(5,1))  AS temp_c,
    trim(diagnosis)                                             AS diagnosis,
    trim(treatment)                                             AS treatment,
    CAST(next_Visit AS DATE)                                    AS next_visit_date,
    current_timestamp()                                         AS dwh_load_date
FROM bi.hospital.medicalrecord
WHERE record_Id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY record_Id ORDER BY visit_Date DESC) = 1;

num_affected_rows,num_inserted_rows


### 1.4 Validación de plata (reconciliación bronce vs. plata)
Si `diferencia` ≠ 0, la deduplicación o el filtro de llaves nulas eliminó filas: hay que investigarlas antes de seguir.

In [0]:
SELECT 'patients' AS tabla, (SELECT count(*) FROM bi.hospital.patients) AS bronce, (SELECT count(*) FROM bi.hospital_silver.patients) AS plata
UNION ALL SELECT 'doctor',        (SELECT count(*) FROM bi.hospital.doctor),        (SELECT count(*) FROM bi.hospital_silver.doctors)
UNION ALL SELECT 'nurse',         (SELECT count(*) FROM bi.hospital.nurse),         (SELECT count(*) FROM bi.hospital_silver.nurses)
UNION ALL SELECT 'helpers',       (SELECT count(*) FROM bi.hospital.helpers),       (SELECT count(*) FROM bi.hospital_silver.helpers)
UNION ALL SELECT 'department',    (SELECT count(*) FROM bi.hospital.department),    (SELECT count(*) FROM bi.hospital_silver.departments)
UNION ALL SELECT 'ward',          (SELECT count(*) FROM bi.hospital.ward),          (SELECT count(*) FROM bi.hospital_silver.wards)
UNION ALL SELECT 'room',          (SELECT count(*) FROM bi.hospital.room),          (SELECT count(*) FROM bi.hospital_silver.rooms)
UNION ALL SELECT 'bed',           (SELECT count(*) FROM bi.hospital.bed),           (SELECT count(*) FROM bi.hospital_silver.beds)
UNION ALL SELECT 'appointment',   (SELECT count(*) FROM bi.hospital.appointment),   (SELECT count(*) FROM bi.hospital_silver.appointments)
UNION ALL SELECT 'bedrecords',    (SELECT count(*) FROM bi.hospital.bedrecords),    (SELECT count(*) FROM bi.hospital_silver.bed_records)
UNION ALL SELECT 'roomrecords',   (SELECT count(*) FROM bi.hospital.roomrecords),   (SELECT count(*) FROM bi.hospital_silver.room_records)
UNION ALL SELECT 'surgeryrecord', (SELECT count(*) FROM bi.hospital.surgeryrecord), (SELECT count(*) FROM bi.hospital_silver.surgery_records)
UNION ALL SELECT 'medicalrecord', (SELECT count(*) FROM bi.hospital.medicalrecord), (SELECT count(*) FROM bi.hospital_silver.medical_records);

tabla,bronce,plata
patients,1500,1500
doctor,400,400
nurse,500,500
helpers,1100,1100
department,31,31
ward,63,63
room,391,391
bed,500,500
appointment,1000,1000
bedrecords,1000,1000


In [0]:
-- Verificación puntual de la limpieza de médicos: ya no debe aparecer "Dr." ni "And" en medio de palabra
SELECT doctor_id, title, full_name, surgeon_type, is_surgeon
FROM bi.hospital_silver.doctors
WHERE full_name RLIKE '(?i)^dr[ .]' OR full_name RLIKE '[a-z]And' OR surgeon_type RLIKE '[a-z]And';
-- Resultado esperado: 0 filas

doctor_id,title,full_name,surgeon_type,is_surgeon


In [0]:
-- Revisión de presión arterial: filas donde el texto existe pero no se pudo separar
SELECT blood_pressure, count(*) AS filas
FROM bi.hospital_silver.medical_records
WHERE blood_pressure IS NOT NULL AND systolic_bp IS NULL
GROUP BY blood_pressure;
-- Resultado esperado: 0 filas. Si aparecen, ajustar el patrón regexp.

blood_pressure,filas
